# OP26 — Phase 0 & 1: Preprocessing
**Agentic Dynamic Tariff Optimization for EV Charging Networks**

This notebook builds the *unified analytical base* for all downstream phases. The heavy lifting lives in `preprocess.py` (importable, tested); this notebook is the narrated, reproducible driver. Run top-to-bottom to regenerate every file in `outputs/`.

The two datasets are kept **separate by design** (different geography, role, and units) — see `ASSUMPTIONS.md`, decision #1.

## Phase 0 — setup
All paths, thresholds, and cleaning rules are centralised in `config.py`. Point the pipeline at the raw OpenProject files via `export OP26_DATA=/path/to/files` or by dropping them in `./data_raw/`.

In [6]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path.cwd().parent))  # repo root: config.py + preprocess.py
import pandas as pd
import config as C
import preprocess as P
pd.set_option('display.max_columns', 50)
print('Reading raw data from:', C.DATA_DIR)

Reading raw data from: C:\Users\rs953\Downloads\OP26_submission_FINAL (1)\op26_submission\data_raw


## Phase 1a — ACN (Caltech) session cleaning
Flatten the JSON-dump spreadsheet, drop wrapper-artifact/null rows, parse RFC-2822 timestamps to Caltech local time, and engineer session features: dwell (`duration_hr`), actual charging (`charging_hr`), post-charge idle/overstay (`idle_hr`), scheduling slack (`laxity_hr`), and `avg_power_kw`. Invalid charging times and implausible power are nulled **and flagged**, never silently dropped.

In [7]:
acn, acn_report = P.clean_acn()
print(f"{len(acn):,} clean sessions | {acn.userID.nunique()} users | {acn.stationID.nunique()} stations")
print(f"with user input (enables laxity): {int(acn.has_user_input.sum()):,}")
acn[['sessionID','connect_local','duration_hr','charging_hr','idle_hr','kWhDelivered','avg_power_kw','laxity_hr','has_user_input']].head()

14,947 clean sessions | 204 users | 54 stations
with user input (enables laxity): 2,225


,sessionID,connect_local,duration_hr,charging_hr,idle_hr,kWhDelivered,avg_power_kw,laxity_hr,has_user_input
0,2_39_78_362_2018-04-25 11:08:04.400812,2018-04-25 04:08:04-07:00,2.201667,2.218333,0.000000,7.932,3.575657,NaN,False
1,2_39_95_27_2018-04-25 13:45:09.617470,2018-04-25 06:45:10-07:00,11.185000,2.984722,8.200278,10.013,3.354751,NaN,False
2,2_39_79_380_2018-04-25 13:45:49.962001,2018-04-25 06:45:50-07:00,9.315278,1.098333,8.216944,5.257,4.786343,NaN,False
3,2_39_79_379_2018-04-25 14:37:06.460772,2018-04-25 07:37:06-07:00,9.307778,1.471111,7.836667,5.177,3.519109,NaN,False
4,2_39_79_381_2018-04-25 14:40:33.638896,2018-04-25 07:40:34-07:00,8.377222,2.998889,5.378333,10.119,3.374250,NaN,False


In [8]:
acn[['duration_hr','charging_hr','idle_hr','laxity_hr','avg_power_kw','kWhDelivered','user_session_count']].describe().T

,count,mean,std,min,25%,50%,75%,max
duration_hr,14947.0,5.680273,4.591983,0.087500,2.019444,4.735556,8.712917,47.588333
charging_hr,14922.0,3.202531,2.862451,0.000000,1.272361,2.242639,4.242708,45.462500
idle_hr,14939.0,2.484662,3.758640,0.000000,0.001389,0.725278,4.139444,46.108056
laxity_hr,2223.0,1.916373,3.095341,-22.606111,0.041528,1.348889,3.831806,11.744444
avg_power_kw,14919.0,3.573328,3.320391,0.038889,2.026067,3.100537,5.414978,218.312035
kWhDelivered,14947.0,8.996258,7.053561,0.501000,4.001068,7.435000,13.204000,69.373000
user_session_count,2225.0,27.407191,18.823048,1.000000,12.000000,22.000000,43.000000,69.000000


## Phase 1b — UrbanEV (Shenzhen) hourly panel
Aggregate the four 5-min matrices (8,640 steps) to **hourly** per zone (occupancy→mean, energy/charging/revenue→sum, price→mean), and engineer the brief's economic features: `utilization = clip(occupancy/piles, 0, 1)`, `revenue`, `revenue_per_kwh`, `occupancy_density`, and a **queue/saturation proxy** (`saturation_count`: 5-min intervals at ≥95% capacity). Calendar features, cyclical encodings, zone metadata (CBD, piles), and lag/rolling features are added per zone (time-ordered, leakage-safe).

In [9]:
panel, zone_features, ev_report = P.build_urbanev_panel()
print(f"panel: {panel.shape[0]:,} rows ({panel.zone.nunique()} zones x {panel.hour_index.nunique()} hours), {panel.shape[1]} columns")
panel[['timestamp','zone','occupancy_mean','capacity','utilization','energy_kwh','price_mean','revenue','saturation_count','util_band']].head()

panel: 177,840 rows (247 zones x 720 hours), 42 columns


,timestamp,zone,occupancy_mean,capacity,utilization,energy_kwh,price_mean,revenue,saturation_count,util_band
0,2022-06-19 00:00:00,102,12.0,30.0,0.4,50.983333,0.924,47.1086,0.0,shoulder
1,2022-06-19 01:00:00,102,12.0,30.0,0.4,52.500000,0.924,48.5100,0.0,shoulder
2,2022-06-19 02:00:00,102,12.0,30.0,0.4,52.500000,0.924,48.5100,0.0,shoulder
3,2022-06-19 03:00:00,102,12.0,30.0,0.4,52.500000,0.924,48.5100,0.0,shoulder
4,2022-06-19 04:00:00,102,12.0,30.0,0.4,52.500000,0.924,48.5100,0.0,shoulder


## Validation & reconciliation
Total energy in the hourly panel must equal the raw 5-min sum exactly, utilization must be in [0,1], and lag NaNs should be warm-up only.

In [10]:
raw_energy = pd.read_csv(C.DATA_DIR / C.F_VOLUME).iloc[:,1:].values.sum()
print('energy reconciliation  raw =', round(raw_energy), ' panel =', round(panel.energy_kwh.sum()),
      ' -> match:', abs(raw_energy - panel.energy_kwh.sum()) < 1)
print('utilization range :', round(panel.utilization.min(),3), '..', round(panel.utilization.max(),3))
print('mean utilization  :', round(panel.utilization.mean(),3))
print('util-band share   :')
print(panel.util_band.value_counts(normalize=True).round(3).to_string())

energy reconciliation  raw = 78274742  panel = 78274742  -> match: True
utilization range : 0.0 .. 1.0
mean utilization  : 0.28
util-band share   :
util_band
offpeak     0.611
shoulder    0.379
peak        0.009


**Key finding that shapes the pricing strategy:** the network is mostly idle — ~61% of zone-hours sit below 30% utilization and only ~1% above 80%. The dominant lever is **discount-driven off-peak uplift**, with targeted surge on the rare hot zones.

In [11]:
# busiest zones by mean utilization (spatial profile for Phase 2 EDA)
zone_features.sort_values('util_mean', ascending=False)[
    ['zone','CBD','dynamic_pricing','capacity','util_mean','util_max','pct_hours_peak','pct_hours_offpeak']].head(8)

,zone,CBD,dynamic_pricing,capacity,util_mean,util_max,pct_hours_peak,pct_hours_offpeak
178,1029,0,1,36.0,0.760269,0.960648,0.276389,0.000000
104,715,0,0,13.0,0.733164,1.000000,0.376389,0.002778
59,570,0,0,21.0,0.639760,1.000000,0.506944,0.270833
110,732,0,1,43.0,0.614578,0.718992,0.000000,0.000000
164,982,0,0,24.0,0.589400,0.875000,0.023611,0.000000
227,1131,0,1,92.0,0.582488,0.849638,0.113889,0.029167
106,719,0,0,6.0,0.579707,1.000000,0.350000,0.215278
133,835,0,0,12.0,0.576659,0.833333,0.006944,0.000000


## Write outputs
Everything downstream reads from these files.

In [12]:
acn.to_csv(C.OUTPUT_DIR / 'clean_acn_sessions.csv', index=False)
panel.to_csv(C.OUTPUT_DIR / 'urbanev_panel_hourly.csv.gz', index=False, compression='gzip')
panel.head(2000).to_csv(C.OUTPUT_DIR / 'urbanev_panel_hourly_sample.csv', index=False)
zone_features.to_csv(C.OUTPUT_DIR / 'zone_features.csv', index=False)
qa = {**acn_report, **ev_report}
pd.DataFrame([{'metric':k,'value':v} for k,v in qa.items()]).to_csv(C.OUTPUT_DIR / 'data_quality_report.csv', index=False)
print('Wrote outputs to', C.OUTPUT_DIR)
for f in sorted(C.OUTPUT_DIR.glob('*')): print('  ', f.name)

Wrote outputs to C:\Users\rs953\Downloads\OP26_submission_FINAL (1)\op26_submission\outputs
   clean_acn_sessions.csv
   data_quality_report.csv
   demand_metrics.csv
   demand_metrics_by_zone.csv
   demand_model_congestion.joblib
   demand_model_energy.joblib
   demand_model_utilization.joblib
   demand_predictions.csv
   eda_acn_behavior.csv
   eda_cbd_comparison.csv
   eda_findings.md
   eda_price_demand_bins.csv
   eda_temporal_profile.csv
   eda_volatility_by_band.csv
   elasticity_estimates.csv
   episode_metrics.csv
   feature_importance.csv
   implications.md
   learned_policy.csv
   monitoring_log.csv
   offpeak_uplift.csv
   peak_windows.csv
   pricing_findings.md
   pricing_policy.csv
   revenue_gain.csv
   robustness_cbd.csv
   robustness_demand_ablation.csv
   robustness_elasticity.csv
   robustness_peak_definition.csv
   robustness_triggers.csv
   tariff_simulation.csv
   urbanev_panel_hourly.csv.gz
   urbanev_panel_hourly_sample.csv
   zone_features.csv


---
**Phase 1 complete.** Next: Phase 2 (EDA + empirical peak-window definition) reads `outputs/urbanev_panel_hourly.csv.gz` and `outputs/clean_acn_sessions.csv`.